
<strong>Nama:</strong> Muhammad Rafly Romeo Nasution<br>
<strong>Kelas:</strong> 4KA25<br>
<strong>NPM:</strong> 10123875

---



# MK Praktikum Unggulan Universitas Gunadarma
# Nama Mata Kuliah: Praktikum Robotika Cerdas (Tingkat 4)  

---
# Pertemuan II

Pada praktikum ini, Anda akan melakukan generate gambar dan GIF menggunakan model GAN (Generative Adversarial Network)


## **1. Mengimport Library Python untuk GAN**

Pada tahapan ini Anda mendefinisikan terlebih dahulu library yang akan Anda gunakan.





In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from keras.datasets import mnist

from tensorflow.keras import Sequential
from keras.layers import BatchNormalization
from keras.layers import Dense, Reshape, Flatten
from keras.layers import LeakyReLU
from tensorflow.keras.optimizers import Adam

import numpy as np
!mkdir generated_images

## **2. Mendefinisikan Variabel untuk Neural Network dan Data**


Pada tahap ini, didefinisikan beberapa variabel yang diperlukan dalam proses pembentukan dan pelatihan model GAN, di antaranya:
 - Ukuran gambar yang akan dihasilkan oleh algoritma GAN.
 - Channel warna yang digunakan pada gambar.
 - Variabel noise (latent) yang menjadi input awal bagi model untuk membentuk gambar.
 - Optimizer, yaitu variabel yang menentukan algoritma optimisasi yang digunakan selama proses pembelajaran. Pada tahap ini, digunakan algoritma Stochastic Gradient Descent (SGD) dengan learning rate sebesar 0.0001. Learning rate menentukan seberapa besar perubahan parameter model pada setiap proses pembelajaran. Nilai learning rate sebesar 0.0001 digunakan sebagai parameter dalam proses pelatihan model GAN.

In [ ]:
img_width = 28
img_height = 28
channels = 1
img_shape = (img_width, img_height, channels)
latent_dim = 100
adam = Adam(learning_rate=0.0001)

## **3. Membentuk Generator**


Generator adalah bagian dari GAN yang bertugas untuk belajar membuat data palsu dengan memasukkan umpan balik dari diskriminator. Generator belajar membuat diskriminator mengklasifikasikan outputnya sebagai yang sebenarnya atau nyata.





In [ ]:
def build_generator():
    model = Sequential()

    model.add(Dense(256, input_dim=latent_dim))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))

    model.add(Dense(256))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))

    model.add(Dense(256))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))

    # Membuat model menjadi ukuran 28x28x1
    model.add(Dense(int(np.prod(img_shape)), activation='tanh'))
    model.add(Reshape(img_shape))

    model.summary()
    return model

In [ ]:
generator = build_generator()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        25,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 784)            │       201,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 28, 28, 1)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 362,000 (1.38 MB)

 Trainable params: 360,464 (1.38 MB)

 Non-trainable params: 1,536 (6.00 KB)

## **4. Membangun Discriminator**

Diskriminator dalam GAN hanyalah sebuah fungsi klasifikasi. Ia mencoba untuk membedakan data nyata dari data yang dibuat oleh generator. Diskriminator bisa menggunakan arsitektur jaringan apa pun yang sesuai dengan jenis data yang diklasifikasikan.

In [ ]:
def build_discriminator():
  model = Sequential()

  model.add(Flatten(input_shape=img_shape))
  model.add(Dense(512))
  model.add(LeakyReLU(alpha=0.2))
  model.add(Dense(256))
  model.add(Dense(1, activation='sigmoid'))

  model.summary()
  return model

discriminator = build_discriminator()
discriminator.compile(loss='binary_crossentropy', optimizer=adam, metrics=['accuracy'])

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 512)            │       401,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 533,505 (2.04 MB)

 Trainable params: 533,505 (2.04 MB)

 Non-trainable params: 0 (0.00 B)

## **5. Menghubungkan Discriminator dan Generator untuk membentuk GAN**



Pada tahapan ini Anda akan menghubungkan discriminator dan generator yang telah dibuat untuk membentuk sebuah GAN.

In [ ]:
GAN = Sequential()
discriminator.trainable = False
GAN.add(generator)
GAN.add(discriminator)

GAN.compile(loss='binary_crossentropy', optimizer=adam)
GAN.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 28, 28, 1)      │       362,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 1)              │       533,505 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 895,505 (3.42 MB)

 Trainable params: 360,464 (1.38 MB)

 Non-trainable params: 535,041 (2.04 MB)

## **6. Menampilkan Output Gambar**

Berikut merupakan langkah untuk menyimpan output gambar. Hasil generate gambar akan tersedia pada folder <code>generated_images</code>.


In [ ]:
import matplotlib.pyplot as plt
import glob
import imageio
import PIL

save_name = 0.00000000

def save_imgs(epoch):
    r, c = 5, 5
    noise = np.random.normal(0, 1, (r * c, latent_dim))
    gen_imgs = generator.predict(noise)
    global save_name
    save_name += 0.00000001
    print("%.8f" % save_name)

    # Rescale images 0 - 1
    gen_imgs = 0.5 * gen_imgs + 0.5

    fig, axs = plt.subplots(r, c)
    cnt = 0
    for i in range(r):
        for j in range(c):
            axs[i,j].imshow(gen_imgs[cnt, :,:,0], cmap='gray')
            # axs[i,j].imshow(gen_imgs[cnt])
            axs[i,j].axis('off')
            cnt += 1
    fig.savefig("generated_images/%.8f.png" % save_name)
    print('saved')
    plt.close()

In [ ]:
save_imgs(1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 966ms/step
0.00000001
saved


## **7. Training GAN**

Model akan melakukan pembelajaran sampai iterasi yang ditentukan. Anda dapat mengubah dan menyesuaikan nilai parameter <code>train</code>.

In [ ]:
def train(epochs, batch_size=64, save_interval=200):
    from keras.datasets import mnist
    import numpy as np
    import matplotlib.pyplot as plt

    # Memastikan variabel global terdefinisi
    global generator, discriminator, GAN, adam, latent_dim, save_imgs

    # Jika save_imgs tidak terdefinisi, buat fallback lokal agar tidak error
    if 'save_imgs' not in globals():
        def save_imgs(epoch):
            r, c = 5, 5
            noise = np.random.normal(0, 1, (r * c, latent_dim))
            gen_imgs = generator.predict(noise, verbose=0)
            global save_name
            if 'save_name' not in globals():
                save_name = 0.00000000
            save_name += 0.00000001
            print("%.8f" % save_name)
            gen_imgs = 0.5 * gen_imgs + 0.5
            fig, axs = plt.subplots(r, c)
            cnt = 0
            for i in range(r):
                for j in range(c):
                    axs[i,j].imshow(gen_imgs[cnt, :,:,0], cmap='gray')
                    axs[i,j].axis('off')
                    cnt += 1
            fig.savefig("generated_images/%.8f.png" % save_name)
            print('saved')
            plt.close()

    # Jika tidak terdefinisi di runtime, definisikan nilai default
    if 'latent_dim' not in globals():
        latent_dim = 100
    if 'adam' not in globals():
        from tensorflow.keras.optimizers import Adam
        adam = Adam(learning_rate=0.0001)

    # Inisialisasi Model jika belum ada
    if 'generator' not in globals() or generator is None:
        generator = build_generator()
    if 'discriminator' not in globals() or discriminator is None:
        discriminator = build_discriminator()
        discriminator.compile(loss='binary_crossentropy', optimizer=adam, metrics=['accuracy'])
    if 'GAN' not in globals() or GAN is None:
        GAN = Sequential()
        discriminator.trainable = False
        GAN.add(generator)
        GAN.add(discriminator)
        GAN.compile(loss='binary_crossentropy', optimizer=adam)

    (X_train, _), (_, _) = mnist.load_data()

    # Rescale data antara -1 dan 1
    X_train = X_train / 127.5 - 1.
    X_train = np.expand_dims(X_train, axis=3)

    # Membuat label target
    valid = np.ones((batch_size, 1))
    fakes = np.zeros((batch_size, 1))

    print("Mulai training...")
    for epoch in range(epochs):
        # Mengambil batch acak
        idx = np.random.randint(0, X_train.shape[0], batch_size)
        imgs = X_train[idx]

        # Menghasilkan gambar buatan (Fake)
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        gen_imgs = generator.predict(noise, verbose=0)

        # Melatih Discriminator
        d_loss_real = discriminator.train_on_batch(imgs, valid)
        d_loss_fake = discriminator.train_on_batch(gen_imgs, fakes)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        # Melatih Generator
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        g_loss = GAN.train_on_batch(noise, valid)

        if (epoch % save_interval) == 0:
            print(f"******* {epoch} [D loss: {d_loss[0]:.4f}, acc: {100 * d_loss[1]:.2f}%] [G loss: {g_loss:.4f}]")
            # Panggil fungsi save_imgs
            save_imgs(epoch)

# Jalankan program pelatihan dengan 3000 iterasi
train(3000, batch_size=64, save_interval=200)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Mulai training...
******* 0 [D loss: 0.6999, acc: 50.39%] [G loss: 0.7370]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
0.00000002
saved
******* 200 [D loss: 1.2753, acc: 32.50%] [G loss: 0.2397]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
0.00000003
saved
******* 400 [D loss: 1.7024, acc: 31.47%] [G loss: 0.1369]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
0.00000004
saved
******* 600 [D loss: 1.9996, acc: 31.28%] [G loss: 0.0956]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
0.00000005
saved
******* 800 [D loss: 2.2243, acc: 31.22%] [G loss: 0.0734]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
0.00000006
saved
******* 1000 [D loss: 2.4047, acc: 31.15%] [G loss: 0.0596]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
0.00000007
saved
******* 1200 [D loss: 2.5535, acc: 31.14%] [G loss: 0.0502]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
0.00000008
saved
******* 1400 [D loss: 2.6810, acc: 31.11%] [G loss: 0.0433]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
0.00000009
saved
******* 1600 [D lo

## **8. Melakukan Generate GIF**

Setelah berhasil membuat gambar, model akan melakukan generate GIF.

In [ ]:
import glob
import imageio
import os

anim_file = 'dcgan.gif'
filenames = glob.glob('generated_images/*.png')
filenames = sorted(filenames)

if len(filenames) == 0:
    print("PERINGATAN: Tidak ada gambar .png ditemukan di folder 'generated_images/'.")
    print("Pastikan fungsi train() sudah dijalankan dan menghasilkan gambar di folder tersebut.")
else:
    with imageio.get_writer(anim_file, mode='I') as writer:
        for filename in filenames:
            image = imageio.v2.imread(filename)
            writer.append_data(image)
        # Menambahkan gambar terakhir sekali lagi agar animasi berhenti sejenak di akhir
        last_image = imageio.v2.imread(filenames[-1])
        writer.append_data(last_image)
    print("GIF berhasil dibuat dengan nama:", anim_file)

GIF berhasil dibuat dengan nama: dcgan.gif


In [ ]:
import os

# Membuat kembali folder generated_images jika belum ada
if not os.path.exists('generated_images'):
    os.makedirs('generated_images')
    print("Folder 'generated_images' berhasil dibuat!")
else:
    print("Folder 'generated_images' sudah ada.")

Folder 'generated_images' sudah ada.


# Instruksi Tugas Praktikum

1. Beri penjelasan dan logika program dari sintaks program yang terdapat pada poin-poin berikut:
   * Poin 6: Menampilkan Output Gambar,
   * Poin 7: Training GAN,
   * Poin 8: Making GIF
2. Silakan modifikasi kode pembelajaran menjadi 3000 parameter iterasi.
3. Apakah terdapat perbedaan dari hasil pembelajaran menggunakan parameter 1500 dan 3000? Jelaskan perbedaannya!


<br>

---

<br>

**CATATAN: DILARANG MENGGUNAKAN AI.** Silakan menjawab pertanyaan dengan analisis sendiri.

1. Penjelasan Logika Program


*   Poin 6: Menampilkan Output Gambar (save_imgs)
Fungsi ini menghasilkan 25 gambar baru menggunakan model Generator yang diberi input berupa noise acak (distribusi normal). Gambar-gambar tersebut kemudian diubah skalanya ke rentang $[0, 1]$$[0, 1]$ agar dapat ditampilkan menggunakan grid $5 \times 5$$5 \times 5$ dari Matplotlib, lalu disimpan otomatis ke folder generated_images/ dengan penamaan file yang urut.
*   Poin 7: Training GAN (train)
Proses ini melatih dua jaringan secara bergantian dalam setiap iterasi. Pertama, melatih Discriminator untuk mengenali mana gambar asli dari dataset MNIST (diberi label 1) dan mana gambar palsu buatan Generator (diberi label 0). Kedua, melatih Generator lewat model gabungan (GAN) dengan label target dibalik menjadi 1 agar Generator belajar menipu Discriminator.
*   Poin 8: Making GIF
Sintaks ini mengambil semua file gambar .png hasil simpanan dari folder generated_images/, mengurutkannya berdasarkan nama file dari iterasi awal hingga akhir, lalu menggabungkannya menjadi satu file animasi .gif menggunakan library imageio.


2. Perbedaan Hasil Pembelajaran (1500 vs 3000 Iterasi)
Jumlah Pembelajaran: Pada parameter 3000 iterasi, model memiliki kesempatan belajar dan memperbarui bobot (weight updates) dua kali lebih banyak dibandingkan dengan 1500 iterasi.
Kualitas Gambar: Hasil gambar pada 3000 iterasi umumnya tampak lebih jelas, detail angka MNIST lebih terbentuk, dan tingkat noise atau buram (blur) berkurang dibandingkan hasil pada 1500 iterasi yang biasanya masih berupa pola goresan kasar.









In [ ]:
# 1. Menghasilkan, menampilkan, dan menyimpan gambar dari Generator, lalu
# melatih Generator dan Discriminator agar Generator semakin mampu membuat gambar menyerupai MNIST
# dan menggabungkan gambar-gambar hasil training menjadi animasi GIF

In [ ]:
# 2. train(3000, batch_size=64, save_interval=200)

In [ ]:
# 3. 3000 iterasi memberikan model kesempatan belajar lebih banyak dibandingkan 1500 iterasi sehingga umumnya kualitas gambar Generator lebih baik, tetapi hasil aktual tetap bergantung pada proses training



---


Copyright © Pengelola MK Praktikum Unggulan (Praktikum DGX), Universitas Gunadarma


https://www.praktikum-hpc.gunadarma.ac.id/